# Notebook 5 — Discovering clue-giving themes (embedding topic modeling)

**MDIG2026 Summer School · NLP track (5 / 5)**

*Topic modeling* asks: across many short texts, what **themes** recur, and which words
characterise each? That's a natural question for our clue-givers — do they fall back on
recurring strategies ("it's a kind of food…", "you use it to…", "it's a place where…")?

The recipe is three steps, all reusing tools you already have in this track:

> **embed** each document → **cluster** the embeddings → label each cluster with its most
> **distinctive words** (a trick called *c-TF-IDF*).

We use lightweight, always-installable pieces (sentence-transformers + scikit-learn), so it runs
in **seconds on any laptop** and reuses the *same* embeddings as the rest of the track.

> Prereq: run **Notebook 1** first to create the transcripts.

In [ ]:
# Use locally-cached models only — never phone home to Hugging Face.
# (The embedding model is already downloaded; this prevents a hang if the
#  network blocks HF's version-check request. Must run before any model load.)
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("Hugging Face offline mode ON — using cached models only.")

## 0 · Install (run once)

In [ ]:
# %pip install sentence-transformers scikit-learn pandas numpy matplotlib

## 1 · Config & documents
Each **clue-giver utterance** is one 'document'.

In [ ]:
import os, certifi
if not os.path.exists(os.environ.get("SSL_CERT_FILE", "")):   # fix broken Anaconda cert path
    os.environ["SSL_CERT_FILE"] = certifi.where(); os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
def find_corpus(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(8):
        if os.path.isfile(os.path.join(d, "metadata.csv")) and os.path.isdir(os.path.join(d, "audios")):
            return d
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    raise FileNotFoundError("Could not find the BalanceCorpus root (metadata.csv + audios/).")
CORPUS = find_corpus()
TRANSCRIPTS = os.path.join(CORPUS, "scripts", "nlp_taboo", "transcripts")
print("Corpus root:", CORPUS)

In [ ]:
import pandas as pd
tx_path = os.path.join(TRANSCRIPTS, "transcripts_all.csv")
if not os.path.exists(tx_path):
    raise FileNotFoundError("Run Notebook 1 first to create transcripts_all.csv.")
tx = pd.read_csv(tx_path, encoding="utf-8")

# keep clue-giver utterances with at least a couple of words
docs = tx[tx.role == "clue_giver"].copy()
docs["text"] = docs["text"].astype(str)
docs = docs[docs.text.str.split().str.len() >= 2].reset_index(drop=True)
if len(docs) < 6:
    raise ValueError("Too few clue-giver utterances for topic modeling. Run Notebook 1 on more "
                     "trials (set MAX_TRIALS = None) and try again.")
print(f"{len(docs)} clue-giver documents from {docs.trial_id.nunique()} trials")
docs[["trial_id", "target_word", "condition", "text"]].head(6)

## 2 · Step 1 — embed the documents
Same multilingual MiniLM model used across the NLP track.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
except ModuleNotFoundError:
    raise ModuleNotFoundError("sentence-transformers is not installed — run the install cell "
                              "in Section 0, then re-run this cell.")
import numpy as np
EMB = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
X = EMB.encode(docs.text.tolist(), normalize_embeddings=True, show_progress_bar=False)
print("embeddings:", X.shape)

## 3 · Step 2 — cluster the embeddings

We use **KMeans** (fast, deterministic, no fragile dependencies). Pick the number of topics
`k`; with few documents keep it small. (On the full corpus you can raise `k`.)

In [ ]:
from sklearn.cluster import KMeans
K = max(2, min(6, len(docs) // 6))           # small-data friendly
labels = KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(X)
docs["topic"] = labels
print(f"{K} topics · sizes: {pd.Series(labels).value_counts().sort_index().to_dict()}")

## 4 · Step 3 — label each topic with its distinctive words (c-TF-IDF)

The trick: treat *all* the documents in a topic as **one big document**, then score each word
by how characteristic it is of that topic versus the others
(`tf × log(1 + average_words / word_frequency)`). The top-scoring words name the topic.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def c_tf_idf(texts, labels, top=8):
    classes = sorted(set(labels))
    joined = [" ".join(t for t, l in zip(texts, labels) if l == c) for c in classes]
    cv = CountVectorizer(stop_words="english", min_df=1)
    C = cv.fit_transform(joined).toarray().astype(float)     # (n_topics, vocab)
    vocab = np.array(cv.get_feature_names_out())
    tf = C / np.maximum(C.sum(axis=1, keepdims=True), 1)     # word rate within a topic
    A = C.sum() / C.shape[0]                                 # avg words per topic
    f = C.sum(axis=0)                                        # total freq of each word
    ctfidf = tf * np.log(1 + A / np.maximum(f, 1))           # distinctive-word score
    return {c: list(vocab[np.argsort(-ctfidf[i])[:top]]) for i, c in enumerate(classes)}

topic_words = c_tf_idf(docs.text.tolist(), labels)
rows = []
for c, ws in topic_words.items():
    ex = docs[docs.topic == c].text.iloc[0]
    rows.append(dict(topic=c, size=int((labels == c).sum()),
                     top_words=", ".join(ws), example=ex[:70]))
pd.DataFrame(rows)

## 5 · Map the topics
PCA to 2-D just for display; colours are the topics we found.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
XY = PCA(n_components=2, random_state=0).fit_transform(X)
cmap = plt.get_cmap("tab10", K)                  # works on all matplotlib versions

fig, ax = plt.subplots(figsize=(8, 6))
for c in range(K):
    m = labels == c
    ax.scatter(XY[m, 0], XY[m, 1], color=cmap(c), s=60, alpha=.8,
               label=f"{c}: {', '.join(topic_words[c][:3])}")
ax.legend(loc="upper left", fontsize=8, frameon=False)
ax.set_title("Clue-giver utterances in embedding space, coloured by topic")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
plt.tight_layout(); plt.show()

## 6 · Do the topics line up with the game?

Two quick checks:
- **Which target words** land in each topic? Topics should gather utterances from
  semantically related targets (a "food" topic pulling doughnut/pasta/bacon, say).
- **Board vs ground:** are some strategies more common under postural load? (Exploratory.)

In [ ]:
print("target words per topic:")
for c in range(K):
    tw = sorted(docs[docs.topic == c].target_word.unique())
    print(f"  topic {c} [{', '.join(topic_words[c][:3])}]: {', '.join(tw)}")

print("\ntopic x condition (row = topic):")
print(pd.crosstab(docs.topic, docs.condition))

## Recap

- We ran a topic model by hand — **embed → cluster → c-TF-IDF** — in seconds, no heavy deps.
- Each topic is named by its most distinctive words, and we checked whether topics recover the
  game's semantic fields and differ by condition.
- Same embeddings as Notebooks 0/2/3, so this sits naturally alongside the rest of the track.

**Caveat:** with only a couple of dyads the topics are coarse — this is exploratory. On the full
corpus (more utterances) the themes sharpen, and you can raise `k`.